# Findings summary — interactive

This notebook loads the paper-facing CSVs (`adversarial/results/paper_*.csv`)
and reproduces the key tables / forest plots in **under 10 seconds**, no API
key required.

Use it to:
1. Spot-check the headline numbers without opening the PDF.
2. Slice the data along axes the paper doesn't print (e.g. per-ticker breakdown).
3. Extend with your own analyses — the CSVs are the source of truth.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS = ROOT / 'adversarial' / 'results'

attacks  = pd.read_csv(RESULTS / 'paper_attack_effects.csv')
defenses = pd.read_csv(RESULTS / 'paper_defense_effects.csv')
arch     = pd.read_csv(RESULTS / 'paper_arch_ablation.csv')

print(f'Loaded:')
print(f'  attacks  : {len(attacks)} rows')
print(f'  defenses : {len(defenses)} rows')
print(f'  arch     : {len(arch)} rows')

## Headline finding — bearish memory poisoning is the only attack that significantly perturbs decisions

Cross-ticker Δ vs clean baseline, with cluster-bootstrap 95 % CI.

In [ ]:
cross_attacks = attacks[attacks['level'] == 'cross'].copy()
cross_attacks['CI'] = cross_attacks.apply(
    lambda r: f"[{r['ci_lo']:+.2f}, {r['ci_hi']:+.2f}]" if pd.notna(r['ci_lo']) else 'n/a',
    axis=1,
)
cross_attacks['Δ'] = cross_attacks['delta'].apply(lambda x: f'{x:+.2f}')
cross_attacks[['batch', 'direction', 'attack', 'Δ', 'CI']]

## Defense effectiveness — single-channel attacks are recoverable; mixed attacks are not

In [ ]:
cross_def = defenses[defenses['level'] == 'cross'].copy()
cross_def['CI'] = cross_def.apply(
    lambda r: f"[{r['ci_lo']:+.2f}, {r['ci_hi']:+.2f}]" if pd.notna(r['ci_lo']) else 'n/a',
    axis=1,
)
cross_def['Δ'] = cross_def['delta'].apply(lambda x: f'{x:+.2f}')
cross_def[['matrix', 'defense', 'attack', 'Δ', 'CI']].head(20)

## Architecture ablation — disabling Bull/Bear debate breaks robustness

In [ ]:
arch_attacked = arch[arch['condition'] != 'clean'].copy()
arch_attacked['Δ'] = arch_attacked['delta_vs_clean'].apply(lambda x: f'{x:+.2f}')
arch_attacked[['variant', 'ticker', 'condition', 'Δ']].head(20)

## Sanity-check: pull the bearish memory-poisoning trial directly from results/

This is the per-trial JSON the headline finding aggregates from.

In [ ]:
import json
from collections import Counter

trials_path = RESULTS / 'campaign' / 'PLTR_2025-12-09_bearish' / 'results_full.json'
with open(trials_path) as f:
    trials = json.load(f)

print(f'PLTR bearish: {len(trials)} trials')
by_cond = Counter(t['condition'] for t in trials)
for cond, n in by_cond.items():
    avg_ord = sum(t['ordinal'] for t in trials if t['condition'] == cond) / n
    print(f'  {cond:>12s}: n={n}, mean ordinal = {avg_ord:.2f}')

## Where to go next

- **Full reproduction guide:** [`adversarial/EXPERIMENTS.md`](../adversarial/EXPERIMENTS.md)
- **Per-batch CSV files:** [`adversarial/results/paper_*.csv`](../adversarial/results/)
- **Statistical methodology:** [`adversarial/stats_hierarchical.py`](../adversarial/stats_hierarchical.py)
- **Paper:** [`paper/5293report.pdf`](../paper/5293report.pdf) §6
- **Static results doc:** [`docs/RESULTS.md`](../docs/RESULTS.md)